# Noise Analysis
What each noise source does to a simulated CHO fed-batch, and reasonable ranges for every setting.

The simulator models noise where it arises physically. Everything defaults to zero, so `NoiseParameters()` is the **noiseless** model; `NoiseParameters.typical()` gives typical levels. Every level can be set on its own, and `simulate()` returns both the **true** trajectory (what happened in the reactor) and the **measured** one (what the assays report).

| source | what it does | where it acts |
|---|---|---|
| **Batch variability** (`batch`) | each run's cells, inoculum and medium differ slightly: kinetic parameters and starting conditions get a lognormal factor once per run | the dynamics |
| **Process noise** (`process`) | growth and production rates fluctuate during the run (Ornstein-Uhlenbeck, a few hours correlation time) | the dynamics |
| **Actuator noise** (`actuator`) | setpoints are missed by a per-run offset, every feed has a dosing error | the dynamics |
| **Measurement noise** (`measurement`) | assays have a per-run calibration bias, per-sample relative errors and a detection limit | the measured values only |

## Reasonable ranges
All levels are fractions (0.05 = 5 %) unless a unit is given. *Low* is a well-controlled process with good analytics, *high* an early-stage process or less precise assays.

**Batch variability** (coefficient of variation between runs)

| setting | low | typical | high | note |
|---|---|---|---|---|
| `batch.mu_max` | 0.02 | 0.05 | 0.10 | growth rate between runs of one cell line |
| `batch.productivity` | 0.05 | 0.10 | 0.20 | specific productivity varies more than growth |
| `batch.q_o2` | 0.05 | 0.10 | 0.20 | specific oxygen uptake |
| `batch.k_death_max` | 0.05 | 0.15 | 0.30 | viability decline is the least reproducible |
| `batch.y_cells_glc` | 0.02 | 0.05 | 0.10 | cell yield on glucose |
| `batch.seeding_density` | 0.03 | 0.07 | 0.15 | inoculum cell count and transfer volume |
| `batch.initial_glucose` | 0.01 | 0.03 | 0.05 | medium batch and preparation |
| `batch.initial_glutamine` | 0.01 | 0.03 | 0.08 | medium batch; glutamine also degrades in storage |

**Process noise** (standard deviation of the log-multiplier on the rate)

| setting | low | typical | high | note |
|---|---|---|---|---|
| `process.growth` | 0.02 | 0.05 | 0.10 | fluctuations of the growth rate |
| `process.production` | 0.02 | 0.05 | 0.10 | fluctuations of the production rate |
| `process.correlation_time` [h] | 4 | 12 | 24 | how long a fluctuation lasts |
| `process.update_interval` [h] | 1 | 2 | 4 | numerical resolution; keep well below the correlation time |

**Actuator noise** (per run, feeds per feed)

| setting | low | typical | high | note |
|---|---|---|---|---|
| `actuator.temperature` [degC] | 0.1 | 0.2 | 0.5 | probe calibration and control offset |
| `actuator.ph` [pH] | 0.01 | 0.03 | 0.05 | pH probe drift |
| `actuator.stirring` | 0.01 | 0.02 | 0.05 | stirring rate error |
| `actuator.feed` | 0.01 | 0.02 | 0.05 | dosing error of a bolus feed |

**Measurement noise**: `relative` is the per-sample error, `absolute` the per-sample error in the variable's unit (three times it is the detection limit), `calibration` a per-run bias shared by all samples of the assay. The cell counter measures total cells and viability (viable = total x viability). For purity the errors act on the impurity fraction (1 - purity). Set `censor_below_detection=True` to report values below the detection limit as NaN.

| variable | relative: low / typical / high | absolute (typical) | calibration (typical) | typical assay |
|---|---|---|---|---|
| `x_total` | 0.05 / 0.07 / 0.10 | 2e4 cells/mL | 0.03 | automated cell counter |
| `viability` | 0.01 / 0.03 / 0.05 | 0 | 0.01 | trypan blue exclusion |
| `c_glc`, `c_lac` | 0.02 / 0.03 / 0.05 | 0.1 / 0.2 mM | 0.02 | enzymatic biochemistry analyser |
| `c_gln`, `c_ammonia` | 0.03 / 0.05 / 0.10 | 0.05 / 0.1 mM | 0.03 | biochemistry analyser |
| `c_product` | 0.03 / 0.05 / 0.10 | 1e-4 mM (~15 mg/L) | 0.03 | Protein A HPLC, bio-layer interferometry |
| `c_aggregates`, `c_glycated` | 0.05 / 0.10 / 0.15 | 1e-4 mM | 0.05 | SEC, boronate / LC-MS |
| `c_hcp` | 0.15 / 0.20 / 0.30 | 1e-4 mM | 0.10 | HCP ELISA |
| `c_o2` | 0.01 / 0.02 / 0.05 | 0.002 mM | 0.02 | optical DO probe |
| `c_co2` | 0.02 / 0.03 / 0.05 | 0.03 mM | 0.02 | blood-gas analyser |
| `purity` (on 1 - purity) | 0.02 / 0.05 / 0.10 | 0.005 | 0.03 | SEC / CE-SDS |

In [1]:
from dataclasses import replace

import numpy as np
import plotly.graph_objects as go

from dbiolab.mammalian_cellculture_simulation import (
    ActuatorNoise,
    BatchVariability,
    CellType,
    FeedConcentrations,
    FeedingEvent,
    MammalianCellcultureReactor,
    MammalianRecipe,
    MeasurementNoise,
    NoiseParameters,
    ProcessNoise,
    ReactorType,
)
from dbiolab.mammalian_cellculture_simulation.scale_library import BASE_CELLTYPE

N_RUNS = 30  # runs per ensemble

## Setup
The tutorial's best recipe in the 2 L stirred tank, with the base CHO cell line.

In [2]:
reactor = ReactorType(
    name="bench_2L",
    volume=2.0,
    scale_factor=1.0,
    growth_inhibition=1.0,
    tank_diameter=0.13,
    impeller_diameter=0.05,
    power_number=1.5,
    superficial_gas_velocity=1.25e-4,
    kla_constant=0.002,
    kla_power_exponent=0.7,
    kla_gas_exponent=0.2,
    max_to_mean_dissipation=30.0,
    o2_saturation=0.2,
    gas_co2_fraction=0.05,
)
simulator = MammalianCellcultureReactor(reactor)
celltype = CellType.from_dict(BASE_CELLTYPE)


def feed(time_point, c_glc, c_gln, temperature, ph_value, stirring_rate):
    amounts = {key: 0.0 for key in FeedConcentrations.__dataclass_fields__}
    amounts.update(c_glc=c_glc, c_gln=c_gln)
    return FeedingEvent(
        time_point=time_point,
        feed_concentrations=FeedConcentrations(**amounts),
        volume_fraction=c_glc / 1000.0 + c_gln / 200.0,
        temperature=temperature,
        ph_value=ph_value,
        stirring_rate=stirring_rate,
    )


def recipe(noise: NoiseParameters) -> MammalianRecipe:
    initial = {key: 0.0 for key in FeedConcentrations.__dataclass_fields__}
    initial.update(
        x_total=8e5, x_viable=8e5, c_glc=5.02, c_gln=1.27, c_o2=0.2, c_co2=1.228
    )
    return MammalianRecipe(
        name="best recipe",
        celltype=celltype,
        temperature=36.13,
        ph_value=6.89,
        stirring_rate=279.78,
        initial_conditions=FeedConcentrations(**initial),
        feeding_strategy=[
            feed(72.0, 1.89, 1.1, 35.68, 6.86, 314.64),
            feed(144.0, 12.39, 4.71, 34.4, 6.85, 349.25),
            feed(216.0, 20.0, 4.97, 33.11, 6.92, 385.89),
        ],
        runtime=336.0,
        measurement_intervals=12.0,
        noise_parameters=noise,
        do_setpoint=0.4,
    )


def ensemble(noise: NoiseParameters, n: int = N_RUNS):
    return [simulator.simulate(recipe(noise), seed=seed) for seed in range(n)]


def band_figure(runs, attribute, title, unit, which="true"):
    values = np.array([getattr(getattr(r, which), attribute) for r in runs])
    t = runs[0].true.timepoints
    mean, std = values.mean(axis=0), values.std(axis=0)
    figure = go.Figure(
        [
            go.Scatter(x=t, y=mean + 2 * std, line={"width": 0}, showlegend=False),
            go.Scatter(
                x=t,
                y=mean - 2 * std,
                fill="tonexty",
                line={"width": 0},
                name="mean +/- 2 sd",
            ),
            go.Scatter(x=t, y=mean, name="mean", line={"color": "black"}),
        ]
    )
    figure.update_layout(
        title=title, xaxis_title="Time (hours)", yaxis_title=unit, width=800, height=400
    )
    return figure

## 1. Noiseless vs. Typical Noise
Without noise, `true` and `measured` are identical and every seed gives the same run. With typical noise, the true trajectory moves (batch, process and actuator noise) and the measurements scatter around it.

In [3]:
noiseless = simulator.simulate(recipe(NoiseParameters()), seed=0)
noisy = simulator.simulate(recipe(NoiseParameters.typical()), seed=0)

figure = go.Figure()
figure.add_trace(
    go.Scatter(
        x=noiseless.true.timepoints,
        y=noiseless.true.c_product,
        name="noiseless",
        line={"color": "black", "dash": "dash"},
    )
)
figure.add_trace(
    go.Scatter(
        x=noisy.true.timepoints,
        y=noisy.true.c_product,
        name="typical noise: true",
        line={"color": "green"},
    )
)
figure.add_trace(
    go.Scatter(
        x=noisy.measured.timepoints,
        y=noisy.measured.c_product,
        name="typical noise: measured",
        mode="markers",
        marker={"color": "green"},
    )
)
figure.update_layout(
    title="Product: Noiseless vs. Typical Noise",
    xaxis_title="Time (hours)",
    yaxis_title="Product (mM)",
    width=800,
    height=400,
)
figure.show()
print(
    "noiseless: true == measured:",
    np.array_equal(noiseless.true.c_product, noiseless.measured.c_product),
)

noiseless: true == measured: True


## 2. Each Source on Its Own
Ensembles of runs with one source at its typical level and everything else off. The table shows how much the final titre varies between runs (coefficient of variation) and how far its mean moves from the noiseless value.

In [4]:
typical = NoiseParameters.typical()
sources = {
    "batch": NoiseParameters(batch=typical.batch),
    "process": NoiseParameters(process=typical.process),
    "actuator": NoiseParameters(actuator=typical.actuator),
    "measurement": NoiseParameters(measurement=typical.measurement),
    "all (typical)": typical,
}
reference = noiseless.true.c_product[-1]
ensembles = {name: ensemble(noise) for name, noise in sources.items()}

print(f"{'source':<15}{'final titre CV':>16}{'mean vs noiseless':>20}")
for name, runs in ensembles.items():
    which = "measured" if name in ("measurement", "all (typical)") else "true"
    final = np.array([getattr(r, which).c_product[-1] for r in runs])
    print(
        f"{name:<15}{final.std() / final.mean():>16.1%}"
        f"{final.mean() / reference - 1:>+20.1%}"
    )

source           final titre CV   mean vs noiseless
batch                      6.3%               -0.7%
process                    1.2%               -0.1%
actuator                   0.8%               -0.7%
measurement                5.3%               -0.8%
all (typical)              9.4%               -2.7%


In [5]:
for name in ("batch", "process", "actuator"):
    band_figure(
        ensembles[name],
        "c_product",
        f"Product, {name} noise only (true)",
        "Product (mM)",
    ).show()
band_figure(
    ensembles["all (typical)"],
    "x_viable",
    "Viable cells, all sources (measured)",
    "Cells (cells/mL)",
    which="measured",
).show()

## 3. Measurement Noise Is Unbiased
Measured over true values, for samples well above the detection limit: centred on 1 over many runs, with a spread from the per-sample error plus the per-run calibration bias. The spread is a little above the relative error where the absolute error (detection limit) still adds to it.

In [6]:
runs = ensembles["measurement"]
for variable in ("x_total", "c_glc", "c_product"):
    true = np.array([getattr(r.true, variable) for r in runs])
    measured = np.array([getattr(r.measured, variable) for r in runs])
    above = true > 10 * typical.measurement.absolute[variable]
    ratio = measured[above] / true[above]
    print(
        f"{variable:<10} mean ratio {ratio.mean():.3f}, spread {ratio.std():.3f} "
        f"(relative {typical.measurement.relative[variable]:.3f}, "
        f"calibration {typical.measurement.calibration[variable]:.3f})"
    )

true = np.array([r.true.c_product for r in runs])
measured = np.array([r.measured.c_product for r in runs])
above = true > 10 * typical.measurement.absolute["c_product"]
ratio = measured[above] / true[above]
go.Figure(go.Histogram(x=ratio, nbinsx=40)).update_layout(
    title="Product: Measured / True",
    xaxis_title="ratio",
    width=800,
    height=350,
).show()

x_total    mean ratio 0.996, spread 0.077 (relative 0.070, calibration 0.030)
c_glc      mean ratio 1.002, spread 0.048 (relative 0.030, calibration 0.020)
c_product  mean ratio 0.995, spread 0.060 (relative 0.050, calibration 0.030)


## 4. How Strong Should the Noise Be?
All sources scaled together with `NoiseParameters.typical().scaled(factor)`. Reported run-to-run variability of the final titre in well-controlled CHO fed-batch processes is roughly 5-15 % (CV); typical levels land in that range.

In [7]:
factors = [0.0, 0.5, 1.0, 2.0]
cvs = []
for factor in factors:
    runs = ensemble(typical.scaled(factor), n=20)
    final = np.array([r.measured.c_product[-1] for r in runs])
    cvs.append(final.std() / final.mean())
    print(f"factor {factor:>3}: final titre CV {cvs[-1]:.1%}")

go.Figure(
    go.Scatter(x=factors, y=np.array(cvs) * 100, mode="lines+markers")
).update_layout(
    title="Run-to-Run Variability vs. Noise Level",
    xaxis_title="noise factor (1 = typical)",
    yaxis_title="final titre CV (%)",
    width=800,
    height=350,
).show()

factor 0.0: final titre CV 0.0%
factor 0.5: final titre CV 5.4%
factor 1.0: final titre CV 11.0%
factor 2.0: final titre CV 22.2%


## 5. Setting Your Own Noise
Start from `NoiseParameters()` (noiseless) or `NoiseParameters.typical()` and change single levels. Each source draws from its own random stream, so switching one off leaves the others' draws unchanged.

In [8]:
custom = NoiseParameters(
    batch=BatchVariability(productivity=0.15),  # only productivity varies
    process=ProcessNoise(growth=0.03, production=0.03, correlation_time=8.0),
    actuator=ActuatorNoise(temperature=0.3, feed=0.03),
    measurement=MeasurementNoise(
        relative={"c_product": 0.08, "x_total": 0.05, "viability": 0.02},
        calibration={"c_product": 0.04},
    ),
)
custom_runs = ensemble(custom, n=10)
final = np.array([r.measured.c_product[-1] for r in custom_runs])
print(f"custom noise: final titre {final.mean():.4f} +/- {final.std():.4f} mM")

# Switching off measurement noise leaves the true runs unchanged
without = replace(custom, measurement=MeasurementNoise())
same = simulator.simulate(recipe(without), seed=0).true.c_product
print("true runs unchanged:", np.array_equal(same, custom_runs[0].true.c_product))

custom noise: final titre 0.0529 +/- 0.0066 mM
true runs unchanged: True
